<a href="https://colab.research.google.com/github/Karthikreddy1010/Electric-poles-and-wires-detection/blob/main/lines_detection_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/Images.zip

Archive:  /content/Images.zip
   creating: New folder/
  inflating: New folder/pano_40.85610527483802_-74.18528095912373_heading_203.26142465064032.jpg  
  inflating: New folder/pano_40.85610776512571_-74.1886854579864_heading_296.98875916015953.jpg  
  inflating: New folder/pano_40.8565230025997_-74.1862866136347_heading_202.17089563395234.jpg  
  inflating: New folder/pano_40.8579626842444_-74.20234291559493_heading_190.95854249318563.jpg  
  inflating: New folder/pano_40.8581791835348_-74.20202144891394_heading_262.1039566246003.jpg  
  inflating: New folder/pano_40.8597838346989_-74.18960528850046_heading_250.92744561052928.jpg  
  inflating: New folder/pano_40.8599152958531_-74.18965074054094_heading_250.92744561052928.jpg  
  inflating: New folder/pano_40.861122800254_-74.19506083089415_heading_124.42886566421123.jpg  
  inflating: New folder/pano_40.8615375047797_-74.18928858398897_heading_214.18124472768432.jpg  
  inflating: New folder/pano_40.8627784326771_-74.18914198280532_

In [ ]:
!pip install ultralytics
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.7 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import math
import json
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import glob

class GSVPoleDetector:
    def __init__(self, output_dir="gsv_analysis_output"):
        self.output_dir = output_dir
        self.camera_height = 2.5
        self.sensor_height = 4.8
        self.focal_length = 5.4

        # Create output directories
        self.create_output_dirs()

        # Load model
        self.model = YOLO('yolov8n-seg.pt')

    def create_output_dirs(self):
        """Create directory structure for output formats"""
        directories = {
            'masks': 'masks',
            'bounding_boxes': 'bounding_boxes',
            'visualizations': 'visualizations',
            'json_data': 'json_data',
            'combined': 'combined'
        }

        # Create main output directory
        os.makedirs(self.output_dir, exist_ok=True)

        # Create subdirectories
        for key, path in directories.items():
            full_path = os.path.join(self.output_dir, path)
            os.makedirs(full_path, exist_ok=True)
            setattr(self, f'{key}_dir', full_path)

    def analyze_images_from_folder(self, folder_path, image_extensions=None):
        """Analyze all images in a folder"""
        if image_extensions is None:
            image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']

        # Get all image paths from folder
        image_paths = []
        for extension in image_extensions:
            image_paths.extend(glob.glob(os.path.join(folder_path, extension)))
            image_paths.extend(glob.glob(os.path.join(folder_path, extension.upper())))

        # Remove duplicates and sort
        image_paths = sorted(list(set(image_paths)))

        if not image_paths:
            print(f"No images found in folder: {folder_path}")
            return {}

        print(f"Found {len(image_paths)} images in folder: {folder_path}")
        return self.analyze_and_export(image_paths)

    def analyze_and_export(self, image_paths):
        """Main analysis function for multiple images"""
        all_results = {}

        for i, image_path in enumerate(image_paths):
            if not os.path.exists(image_path):
                print(f"Image not found: {image_path}")
                continue

            image_name = os.path.splitext(os.path.basename(image_path))[0]
            print(f"Processing {i+1}/{len(image_paths)}: {image_name}")

            image = cv2.imread(image_path)
            if image is None:
                print(f"Could not load image: {image_path}")
                continue

            # Process image and get all detections
            results = self.process_image_detections(image, image_name)

            # Export all formats
            self.export_all_formats(image, results, image_name)

            all_results[image_name] = results

        # Generate combined report
        self.generate_combined_report(all_results)
        return all_results

    def process_image_detections(self, image, image_name):
        """Process image and return all detection formats"""
        height, width = image.shape[:2]

        # Detect poles with segmentation
        pole_results = self.model(image)
        poles = self.extract_pole_detections(pole_results, width, height)

        # Calculate heights
        heights = self.calculate_pole_heights(poles, height)

        return {
            'image_size': (width, height),
            'poles': poles,
            'heights': heights,
            'image_name': image_name
        }

    def extract_pole_detections(self, results, img_width, img_height):
        """Extract pole detections with masks and bounding boxes"""
        poles = []

        for r in results:
            if r.masks is not None:
                for i, (mask, box) in enumerate(zip(r.masks.data, r.boxes)):
                    confidence = box.conf[0].cpu().numpy()

                    if confidence > 0.3:
                        # Bounding box coordinates
                        bbox = box.xyxy[0].cpu().numpy()

                        # Segmentation mask
                        mask_array = mask.cpu().numpy()

                        # Convert mask to binary and resize to original image size
                        binary_mask = (mask_array > 0.5).astype(np.uint8) * 255

                        pole_data = {
                            'id': i,
                            'bbox': bbox.tolist(),  # [x1, y1, x2, y2]
                            'mask': binary_mask,
                            'confidence': float(confidence),
                            'class_id': int(box.cls[0].cpu().numpy())
                        }
                        poles.append(pole_data)

        return poles

    def calculate_pole_heights(self, poles, image_height):
        """Calculate pole heights using GSV geometry"""
        heights = {}

        for i, pole in enumerate(poles):
            mask = pole['mask']
            coords = np.where(mask > 0)

            if len(coords[0]) > 0:
                pole_top = np.min(coords[0])
                pole_bottom = np.max(coords[0])

                height = self.calculate_height_geometry(pole_bottom, pole_top, image_height)

                heights[f'pole_{i}'] = {
                    'height_meters': height,
                    'pixel_height': int(pole_bottom - pole_top),
                    'confidence': pole['confidence']
                }

        return heights

    def calculate_height_geometry(self, bottom_y, top_y, image_height):
        """Calculate real-world height from pixel coordinates"""
        fov_vertical = 2 * math.atan(self.sensor_height / (2 * self.focal_length))
        theta_bottom = (bottom_y / image_height - 0.5) * fov_vertical
        theta_top = (top_y / image_height - 0.5) * fov_vertical

        distance = self.camera_height / math.tan(abs(theta_bottom))
        pole_height = distance * math.tan(abs(theta_top)) - self.camera_height

        return abs(pole_height)

    def export_all_formats(self, image, results, image_name):
        """Export all data formats for a single image"""

        # 1. Export Masks
        self.export_masks(image, results, image_name)

        # 2. Export Bounding Boxes
        self.export_bounding_boxes(results, image_name)

        # 3. Export JSON Data
        self.export_json_data(results, image_name)

        # 4. Export Combined Visualization
        self.export_combined_visualization(image, results, image_name)

        # 5. Export Individual Visualizations
        self.export_individual_visualizations(image, results, image_name)

    def export_masks(self, image, results, image_name):
        """Export segmentation masks as PNG images"""
        mask_dir = os.path.join(self.masks_dir, image_name)
        os.makedirs(mask_dir, exist_ok=True)

        # Export individual pole masks
        for i, pole in enumerate(results['poles']):
            mask = pole['mask']
            mask_path = os.path.join(mask_dir, f'pole_{i}_mask.png')
            cv2.imwrite(mask_path, mask)

        # Export combined mask
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for pole in results['poles']:
            combined_mask = cv2.bitwise_or(combined_mask, pole['mask'])

        combined_mask_path = os.path.join(self.masks_dir, f'{image_name}_combined_mask.png')
        cv2.imwrite(combined_mask_path, combined_mask)

        print(f"  Masks exported to: {mask_dir}")

    def export_bounding_boxes(self, results, image_name):
        """Export bounding boxes in multiple formats"""
        bbox_data = {
            'image_name': image_name,
            'image_size': results['image_size'],
            'poles': []
        }

        for i, pole in enumerate(results['poles']):
            bbox_info = {
                'id': i,
                'bbox': pole['bbox'],  # [x1, y1, x2, y2]
                'confidence': pole['confidence'],
                'height_meters': results['heights'].get(f'pole_{i}', {}).get('height_meters', 0)
            }
            bbox_data['poles'].append(bbox_info)

        # Export as JSON
        json_path = os.path.join(self.bounding_boxes_dir, f'{image_name}_bboxes.json')
        with open(json_path, 'w') as f:
            json.dump(bbox_data, f, indent=2)

        # Export as YOLO format (normalized coordinates)
        yolo_path = os.path.join(self.bounding_boxes_dir, f'{image_name}_bboxes.txt')
        self.export_yolo_format(bbox_data, yolo_path, results['image_size'])

        print(f"  Bounding boxes exported to: {json_path}")

    def export_yolo_format(self, bbox_data, output_path, image_size):
        """Export bounding boxes in YOLO format (normalized coordinates)"""
        img_w, img_h = image_size

        with open(output_path, 'w') as f:
            for pole in bbox_data['poles']:
                x1, y1, x2, y2 = pole['bbox']

                # Convert to normalized coordinates
                x_center = ((x1 + x2) / 2) / img_w
                y_center = ((y1 + y2) / 2) / img_h
                width = (x2 - x1) / img_w
                height = (y2 - y1) / img_h

                # YOLO format: class x_center y_center width height
                line = f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n"
                f.write(line)

    def export_json_data(self, results, image_name):
        """Export complete analysis data as JSON"""
        complete_data = {
            'metadata': {
                'image_name': image_name,
                'image_size': results['image_size'],
                'camera_parameters': {
                    'height': self.camera_height,
                    'sensor_height': self.sensor_height,
                    'focal_length': self.focal_length
                }
            },
            'detections': {
                'poles_count': len(results['poles']),
                'poles': [],
                'heights': results['heights']
            }
        }

        # Add detailed pole information
        for i, pole in enumerate(results['poles']):
            pole_info = {
                'id': i,
                'bbox': pole['bbox'],
                'confidence': pole['confidence'],
                'class_id': pole['class_id'],
                'height_meters': results['heights'].get(f'pole_{i}', {}).get('height_meters', 0),
                'pixel_height': results['heights'].get(f'pole_{i}', {}).get('pixel_height', 0)
            }
            complete_data['detections']['poles'].append(pole_info)

        json_path = os.path.join(self.json_data_dir, f'{image_name}_complete_analysis.json')
        with open(json_path, 'w') as f:
            json.dump(complete_data, f, indent=2)

        print(f"  Complete JSON data exported to: {json_path}")

    def export_combined_visualization(self, image, results, image_name):
        """Create combined visualization with all detections"""
        viz_image = image.copy()

        # Draw masks with transparency
        for pole in results['poles']:
            mask = pole['mask']
            colored_mask = np.zeros_like(viz_image)
            colored_mask[mask > 0] = [0, 255, 0]  # Green
            viz_image = cv2.addWeighted(viz_image, 1, colored_mask, 0.3, 0)

        # Draw bounding boxes
        for i, pole in enumerate(results['poles']):
            bbox = [int(coord) for coord in pole['bbox']]
            cv2.rectangle(viz_image, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)

            # Add label with height
            height_info = results['heights'].get(f'pole_{i}', {})
            if height_info:
                label = f"Pole: {height_info['height_meters']:.1f}m (conf: {pole['confidence']:.2f})"
                cv2.putText(viz_image, label, (bbox[0], bbox[1]-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        output_path = os.path.join(self.combined_dir, f'{image_name}_combined_visualization.jpg')
        cv2.imwrite(output_path, viz_image)

        print(f"  Combined visualization exported to: {output_path}")

    def export_individual_visualizations(self, image, results, image_name):
        """Create individual visualizations for each format"""
        vis_dir = os.path.join(self.visualizations_dir, image_name)
        os.makedirs(vis_dir, exist_ok=True)

        # 1. Bounding Boxes only
        bbox_viz = image.copy()
        for i, pole in enumerate(results['poles']):
            bbox = [int(coord) for coord in pole['bbox']]
            cv2.rectangle(bbox_viz, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 3)

            # Add confidence label
            label = f"Conf: {pole['confidence']:.2f}"
            cv2.putText(bbox_viz, label, (bbox[0], bbox[1]-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.imwrite(os.path.join(vis_dir, 'bounding_boxes_only.jpg'), bbox_viz)

        # 2. Masks only
        mask_viz = image.copy()
        for pole in results['poles']:
            mask = pole['mask']
            colored_mask = np.zeros_like(mask_viz)
            colored_mask[mask > 0] = [0, 255, 0]
            mask_viz = cv2.addWeighted(mask_viz, 1, colored_mask, 0.5, 0)
        cv2.imwrite(os.path.join(vis_dir, 'masks_only.jpg'), mask_viz)

    def generate_combined_report(self, all_results):
        """Generate a comprehensive report for all images"""
        report = {
            'analysis_summary': {
                'total_images': len(all_results),
                'total_poles': 0,
                'average_pole_height': 0,
                'average_confidence': 0
            },
            'image_details': {}
        }

        total_height = 0
        total_confidence = 0
        pole_count = 0

        for img_name, results in all_results.items():
            poles_in_image = len(results['poles'])

            report['image_details'][img_name] = {
                'poles_count': poles_in_image,
                'pole_heights': results['heights']
            }

            report['analysis_summary']['total_poles'] += poles_in_image

            # Calculate average height and confidence
            for i, pole in enumerate(results['poles']):
                height_info = results['heights'].get(f'pole_{i}', {})
                if height_info:
                    total_height += height_info['height_meters']
                    total_confidence += pole['confidence']
                    pole_count += 1

        if pole_count > 0:
            report['analysis_summary']['average_pole_height'] = total_height / pole_count
            report['analysis_summary']['average_confidence'] = total_confidence / pole_count

        # Save report
        report_path = os.path.join(self.output_dir, 'combined_analysis_report.json')
        with open(report_path, 'w') as f:
            json.dump(report, f, indent=2)

        # Print summary
        print("\n" + "="*60)
        print("ANALYSIS COMPLETE - OUTPUT SUMMARY")
        print("="*60)
        print(f"Output directory: {self.output_dir}")
        print(f"Total images processed: {report['analysis_summary']['total_images']}")
        print(f"Total poles detected: {report['analysis_summary']['total_poles']}")
        print(f"Average pole height: {report['analysis_summary']['average_pole_height']:.2f} meters")
        print(f"Average confidence: {report['analysis_summary']['average_confidence']:.2f}")
        print("\nOutput formats generated for each image:")
        print("  ✓ Segmentation masks (PNG)")
        print("  ✓ Bounding boxes (JSON, YOLO format)")
        print("  ✓ Complete analysis data (JSON)")
        print("  ✓ Visualization images")
        print("  ✓ Combined reports")
        print("="*60)

# Usage Examples
def main():
    """Main function with multiple usage options"""

    # OPTION 1: Process all images in a folder
    folder_path = "/content/New folder"  # Replace with your folder path

    # Initialize detector
    detector = GSVPoleDetector("my_gsv_analysis")

    # Process folder
    results = detector.analyze_images_from_folder(folder_path)

    return results

def process_folder_interactive():
    """Interactive function to select folder"""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()

    folder_path = filedialog.askdirectory(title="Select folder containing GSV images")

    if folder_path:
        detector = GSVPoleDetector("my_gsv_analysis")
        results = detector.analyze_images_from_folder(folder_path)
        return results
    else:
        print("No folder selected.")
        return {}

def process_specific_images():
    """Process specific image files"""
    image_paths = [
        "image1.jpg",
        "image2.jpg",
        "image3.png"
    ]

    detector = GSVPoleDetector("my_gsv_analysis")
    results = detector.analyze_and_export(image_paths)
    return results

if __name__ == "__main__":
    # Choose one of the following options:

    # Option 1: Hardcoded folder path
    main()

    # Option 2: Interactive folder selection
    # process_folder_interactive()

    # Option 3: Specific image list
    # process_specific_images()

Found 18 images in folder: /content/New folder
Processing 1/18: pano_40.85610527483802_-74.18528095912373_heading_203.26142465064032

0: 640x640 1 traffic light, 140.5ms
Speed: 3.2ms preprocess, 140.5ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
  Masks exported to: my_gsv_analysis/masks/pano_40.85610527483802_-74.18528095912373_heading_203.26142465064032
  Bounding boxes exported to: my_gsv_analysis/bounding_boxes/pano_40.85610527483802_-74.18528095912373_heading_203.26142465064032_bboxes.json
  Complete JSON data exported to: my_gsv_analysis/json_data/pano_40.85610527483802_-74.18528095912373_heading_203.26142465064032_complete_analysis.json
  Combined visualization exported to: my_gsv_analysis/combined/pano_40.85610527483802_-74.18528095912373_heading_203.26142465064032_combined_visualization.jpg
Processing 2/18: pano_40.85610776512571_-74.1886854579864_heading_296.98875916015953

0: 640x640 (no detections), 132.6ms
Speed: 2.3ms preprocess, 132.6ms inference, 0

In [ ]:
!zip -r /content/my_gsv_analysis.zip /content/my_gsv_analysis

  adding: content/my_gsv_analysis/ (stored 0%)
  adding: content/my_gsv_analysis/masks/ (stored 0%)
  adding: content/my_gsv_analysis/masks/pano_40.8627784326771_-74.18914198280532_heading_297.75854060141324_combined_mask.png (deflated 71%)
  adding: content/my_gsv_analysis/masks/pano_40.8627784326771_-74.18914198280532_heading_297.75854060141324/ (stored 0%)
  adding: content/my_gsv_analysis/masks/pano_40.8627784326771_-74.18914198280532_heading_297.75854060141324/pole_0_mask.png (deflated 79%)
  adding: content/my_gsv_analysis/masks/pano_40.8627784326771_-74.18914198280532_heading_297.75854060141324/pole_1_mask.png (deflated 76%)
  adding: content/my_gsv_analysis/masks/pano_40.8634773184374_-74.2028618196771_heading_113.655705053855/ (stored 0%)
  adding: content/my_gsv_analysis/masks/pano_40.8634773184374_-74.2028618196771_heading_113.655705053855/pole_0_mask.png (deflated 89%)
  adding: content/my_gsv_analysis/masks/pano_40.861122800254_-74.19506083089415_heading_124.42886566421123